<a href="https://colab.research.google.com/github/Satya-Padala/24P31A42V7_DriveReady_OPC_2026/blob/main/Day-4/task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Santhi's pickles")      # run once; next time use gc.open("Sweet Crumbs Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "duration", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1xJmQou5ryxk7xQd_UNSa432qRFoIqXRESn50jas8knc


In [26]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"chicken pickle": 1000, "mango pickle": 800, "prawn pickle": 900}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm"}

class Order(BaseModel):
    """A pickle order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["chicken pickle", "mango pickle","prawn pickle"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    duration: int = Field(description="in how many days it should be delivered asked by customer")

In [31]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """does stuff
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the shop opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, duration: int) -> str:
    """Save a confirmed pickle order to the Orders sheet.
    Use only when the customer has given their name, item, quantity,duration"""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, duration=duration)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.duration,total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.duration}, Rs {total}."

In [35]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly assistant for Santhi's pickle shop.
FACTS:  Chicken pickle Rs 1000. Prawn pickle Rs 900. Mango pickle Rs 800 each.
        we prepare it after we get an order;veg and non-veg pickles are available;
        for one order it takes 5-6 days.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Reply in a polite way in customer's language;answer should be under 50 words."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on monday?"))
print(chat("how much cost for 2kg chicken pickles and 3kg mango pickle"))
print(chat("I'm satya. Please book 5kg prawn pickle and deliver it in 10 days."))
print(chat("I'm chinni. Please book 1kg chicken pickle and deliver it in 5 days."))
print(chat("I'm josh. Please book 10kg mango pickle and deliver it in 15 days."))

Yes, we’re open on Monday from 9 am to 8 pm. Feel free to visit or place an order!
Chicken pickle 2 kg costs Rs 2000, mango pickle 3 kg costs Rs 2400. Total: Rs 4400. Orders take 5‑6 days. May I have your name, quantity details, and preferred pickup date to log the order?
✅ Order confirmed!  
5 kg prawn pickle for Satya, pickup in 10 days. Total: Rs 4500.  
We’ll prepare it and have it ready for you. Thank you for choosing our pickle shop!
✅ Order confirmed!  
1 kg chicken pickle for Chinni, pickup in 5 days. Total: Rs 1000.  
We’ll prepare it and have it ready for you. Thank you!
✅ Order confirmed!  
10 kg mango pickle for Josh, pickup in 15 days. Total: Rs 8000.  
We’ll start preparing it right away. Thanks for choosing us!


In [34]:
history = []

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Are you open on Monday?"}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

model_calls = 0
tool_calls = 0

for msg in result["messages"]:
    if msg.type == "ai":
        model_calls += 1
        tool_calls += len(msg.tool_calls)

print("Model calls:", model_calls)
print("Tool calls:", tool_calls)

================================ Human Message =================================

Are you open on Monday?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_4c2e633e-4ef6-4524-9f8d-cce085dd1183)
 Call ID: fc_4c2e633e-4ef6-4524-9f8d-cce085dd1183
  Args:
    day: monday
================================= Tool Message =================================
Name: is_open

Open 9 am to 8 pm on monday.
================================== Ai Message ==================================

Yes! We’re open Monday from 9 am to 8 pm. Let me know if you’d like to place an order.
Model calls: 2
Tool calls: 1


SANthi's PICKLES – BUSINESS FACTS

1. Business name: Santhi's Pickles.
2. Business type: A pickle business that accepts customer orders.
3. Available pickle varieties include chicken pickle.
4. Available pickle varieties include prawn pickle.
5. Available pickle varieties include mango pickle.
6. Chicken pickle costs Rs 1,000 per kg.
7. Prawn pickle costs Rs 900 per kg.
8. Mango pickle costs Rs 800 per kg.
9. Customers can order different quantities of the available pickles.
10. The order total is calculated as the price per kg multiplied by the quantity in kg.
11. Orders are prepared after the customer places an order.
12. Preparing an order takes approximately 5–6 days.
13. The shop is open Monday from 9 AM to 8 PM.
14. The shop is open Tuesday from 9 AM to 8 PM.
15. The shop is open Wednesday from 9 AM to 8 PM.
16. The shop is open Thursday from 9 AM to 8 PM.
17. The shop is open Friday from 9 AM to 8 PM.
18. The shop is open Saturday from 9 AM to 8 PM.
19. Sunday opening hours have not been specified; customers should be asked to confirm.
20. Delivery areas, delivery charges, payment methods, and cancellation/refund policies have not been specified; the assistant must not guess them and should ask the business owner for confirmation.
